In [1]:
import os
import glob
import json
import pickle
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Tuple, Any, Union
from dataclasses import dataclass, asdict

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import MiniBatchKMeans, KMeans
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC, LinearSVC
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    f1_score,
    precision_score,
    recall_score
)


In [2]:
@dataclass
class ExperimentConfig:
    """Configuration for a single experiment"""
    # Data parameters
    data_dir: str = "flowers"
    image_max_size: int = 400
    test_size: float = 0.2
    random_state: int = 42

    # Feature extraction parameters
    sift_n_features: int = 0  # 0 = unlimited
    sift_contrast_threshold: float = 0.04

    # Vocabulary parameters
    vocab_size: int = 150
    use_minibatch_kmeans: bool = True
    kmeans_batch_size: int = 1000

    # Classifier parameters
    knn_neighbors: int = 5
    knn_metric: str = 'euclidean'
    normalize_histograms: bool = True

    # Experiment metadata
    experiment_name: str = "baseline"
    experiment_description: str = ""

    def to_dict(self) -> Dict:
        return asdict(self)


In [3]:
class DataLoader:
    """Handles loading and preprocessing of image data"""

    def __init__(self, config: ExperimentConfig):
        self.config = config

    def load_images_and_labels(self) -> Tuple[List[np.ndarray], np.ndarray, List[str]]:
        """
        Load images from directory structure: data_dir/<class>/*.jpg

        Returns:
            images: List of BGR images (resized)
            labels: Array of integer labels
            class_names: List of class names
        """
        images = []
        labels = []

        class_names = sorted(os.listdir(self.config.data_dir))
        class_to_idx = {c: i for i, c in enumerate(class_names)}

        print(f"Loading images from: {self.config.data_dir}")
        print(f"Found classes: {class_names}")

        for class_name in class_names:
            class_dir = os.path.join(self.config.data_dir, class_name)
            image_files = glob.glob(os.path.join(class_dir, "*"))

            for img_path in image_files:
                img = cv2.imread(img_path)
                if img is None:
                    print(f"Warning: Could not read {img_path}")
                    continue

                # Resize image to limit computation
                img_resized = self._resize_image(img)
                images.append(img_resized)
                labels.append(class_to_idx[class_name])

        print(f"Loaded {len(images)} images from {len(class_names)} classes")
        return images, np.array(labels), class_names

    def _resize_image(self, img: np.ndarray) -> np.ndarray:
        """Resize image maintaining aspect ratio"""
        h, w = img.shape[:2]
        max_dim = max(h, w)

        if max_dim > self.config.image_max_size:
            scale = self.config.image_max_size / max_dim
            new_w, new_h = int(w * scale), int(h * scale)
            img = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)

        return img


In [4]:
class FeatureExtractor:
    """Extracts SIFT features from images"""

    def __init__(self, config: ExperimentConfig):
        self.config = config
        self.sift = cv2.SIFT_create(
            nfeatures=config.sift_n_features,
            contrastThreshold=config.sift_contrast_threshold
        )

    def extract_descriptors(self, images: List[np.ndarray]) -> Tuple[List[np.ndarray], np.ndarray]:
        """
        Extract SIFT descriptors from all images

        Returns:
            image_descriptors: List of descriptor arrays per image (N_i x 128)
            all_descriptors_stacked: All descriptors concatenated (M x 128)
                where N_i = number of keypoints in image i
                      M = total keypoints across all images
        """
        image_descriptors = []
        all_descriptors = []

        print("Extracting SIFT descriptors...")
        for i, img in enumerate(images):
            if (i + 1) % 100 == 0:
                print(f"  Processed {i + 1}/{len(images)} images")

            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            keypoints, descriptors = self.sift.detectAndCompute(gray, None)

            # Handle images with no keypoints
            if descriptors is None:
                descriptors = np.zeros((0, 128), dtype=np.float32)

            image_descriptors.append(descriptors)

            if descriptors.shape[0] > 0:
                all_descriptors.append(descriptors)

        if len(all_descriptors) == 0:
            raise RuntimeError("No descriptors found in any image")

        all_descriptors_stacked = np.vstack(all_descriptors).astype(np.float32)
        print(f"Total descriptors extracted: {all_descriptors_stacked.shape[0]}")

        return image_descriptors, all_descriptors_stacked


In [5]:
class VocabularyBuilder:
    """Builds visual vocabulary using K-Means clustering"""

    def __init__(self, config: ExperimentConfig):
        self.config = config
        self.kmeans = None
        self.vocabulary = None

    def build_vocabulary(self, descriptors: np.ndarray) -> np.ndarray:
        """
        Build visual vocabulary by clustering descriptors

        Args:
            descriptors: All SIFT descriptors (M x 128)

        Returns:
            vocabulary: Cluster centers (vocab_size x 128)
        """
        print(f"Building vocabulary with {self.config.vocab_size} visual words...")

        if self.config.use_minibatch_kmeans:
            self.kmeans = MiniBatchKMeans(
                n_clusters=self.config.vocab_size,
                random_state=self.config.random_state,
                batch_size=self.config.kmeans_batch_size,
                verbose=0
            )
        else:
            self.kmeans = KMeans(
                n_clusters=self.config.vocab_size,
                random_state=self.config.random_state,
                verbose=0
            )

        self.kmeans.fit(descriptors)
        self.vocabulary = self.kmeans.cluster_centers_

        print(f"Vocabulary built. Shape: {self.vocabulary.shape}")
        return self.vocabulary

    def compute_histograms(self, image_descriptors: List[np.ndarray]) -> np.ndarray:
        """
        Compute histogram of visual words for each image

        Args:
            image_descriptors: List of descriptor arrays per image

        Returns:
            histograms: Array of shape (n_images, vocab_size)
        """
        n_images = len(image_descriptors)
        histograms = np.zeros((n_images, self.config.vocab_size), dtype=np.float32)

        print("Computing BoVW histograms...")
        for i, descriptors in enumerate(image_descriptors):
            if descriptors.shape[0] == 0:
                # No features detected in this image
                continue

            # Assign each descriptor to nearest visual word
            visual_words = self.kmeans.predict(descriptors)

            # Create histogram
            hist, _ = np.histogram(
                visual_words,
                bins=np.arange(self.config.vocab_size + 1)
         )

            # Normalize histogram (L2 normalization)
            if self.config.normalize_histograms and hist.sum() > 0:
                hist = hist.astype(np.float32)
                hist = hist / np.linalg.norm(hist)

            histograms[i] = hist

        return histograms


In [6]:
class Classifier:
    """Handles classification using KNN"""

    def __init__(self, config: ExperimentConfig):
        self.config = config
        self.model = None

    def train(self, X_train: np.ndarray, y_train: np.ndarray):
        """Train KNN classifier"""
        print(f"Training KNN classifier (k={self.config.knn_neighbors})...")

        self.model = KNeighborsClassifier(
            n_neighbors=self.config.knn_neighbors,
            metric=self.config.knn_metric
        )
        self.model.fit(X_train, y_train)

    def predict(self, X: np.ndarray) -> np.ndarray:
        """Make predictions"""
        return self.model.predict(X)

    def evaluate(self, X_test: np.ndarray, y_test: np.ndarray,
                 class_names: List[str]) -> Dict[str, Any]:
        """
        Evaluate model and return metrics

        Returns:
            metrics: Dictionary containing various performance metrics
        """
        y_pred = self.predict(X_test)

        metrics = {
            'accuracy': accuracy_score(y_test, y_pred),
            'precision': precision_score(y_test, y_pred, average='weighted'),
            'recall': recall_score(y_test, y_pred, average='weighted'),
            'f1_score': f1_score(y_test, y_pred, average='weighted'),
            'confusion_matrix': confusion_matrix(y_test, y_pred),
            'classification_report': classification_report(
                y_test, y_pred,
                target_names=class_names,
                output_dict=True
            )
        }

        return metrics


In [7]:
class ExperimentRunner:
    """Runs complete experiment pipeline"""

    def __init__(self, config: ExperimentConfig):
        self.config = config
        self.results = {}

    def run(self) -> Dict[str, Any]:
        """Execute complete experiment pipeline"""
        start_time = datetime.now()
        print(f"\n{'='*60}")
        print(f"Running Experiment: {self.config.experiment_name}")
        print(f"{'='*60}\n")

        # 1. Load data
        data_loader = DataLoader(self.config)
        images, labels, class_names = data_loader.load_images_and_labels()

        # 2. Extract features
        feature_extractor = FeatureExtractor(self.config)
        image_descriptors, all_descriptors = feature_extractor.extract_descriptors(images)

        # 3. Build vocabulary
        vocab_builder = VocabularyBuilder(self.config)
        vocabulary = vocab_builder.build_vocabulary(all_descriptors)

        # 4. Compute histograms
        histograms = vocab_builder.compute_histograms(image_descriptors)

        # 5. Train/test split
        X_train, X_test, y_train, y_test = train_test_split(
            histograms, labels,
            test_size=self.config.test_size,
            random_state=self.config.random_state,
            stratify=labels
        )

        print(f"Train set: {X_train.shape[0]} samples")
        print(f"Test set: {X_test.shape[0]} samples")

        # 6. Train classifier
        classifier = Classifier(self.config)
        classifier.train(X_train, y_train)

        # 7. Evaluate
        metrics = classifier.evaluate(X_test, y_test, class_names)

        # Store results
        end_time = datetime.now()
        duration = (end_time - start_time).total_seconds()

        self.results = {
            'config': self.config.to_dict(),
            'metrics': metrics,
            'class_names': class_names,
            'timestamp': start_time.isoformat(),
            'duration_seconds': duration,
            'data_stats': {
                'n_images': len(images),
                'n_classes': len(class_names),
                'n_train': X_train.shape[0],
                'n_test': X_test.shape[0],
                'n_descriptors': all_descriptors.shape[0]
            }
        }

        # Print summary
        self._print_summary()

        return self.results

    def _print_summary(self):
        """Print experiment summary"""
        print(f"\n{'='*60}")
        print("EXPERIMENT RESULTS")
        print(f"{'='*60}")
        print(f"Experiment: {self.config.experiment_name}")
        print(f"Duration: {self.results['duration_seconds']:.2f} seconds")
        print(f"\nMetrics:")
        print(f"  Accuracy:  {self.results['metrics']['accuracy']:.4f}")
        print(f"  Precision: {self.results['metrics']['precision']:.4f}")
        print(f"  Recall:    {self.results['metrics']['recall']:.4f}")
        print(f"  F1-Score:  {self.results['metrics']['f1_score']:.4f}")
        print(f"{'='*60}\n")


In [8]:
class HyperparameterTuner:
    """Performs hyperparameter tuning using grid search"""

    def __init__(self, base_config: ExperimentConfig):
        self.base_config = base_config
        self.results = []

    def tune(self, param_grid: Dict[str, List]) -> pd.DataFrame:
        """
        Run grid search over parameter combinations

        Args:
            param_grid: Dictionary mapping parameter names to lists of values
                Example: {
                    'vocab_size': [50, 100, 150],
                    'knn_neighbors': [3, 5, 7],
                }

        Returns:
            results_df: DataFrame with all experiment results
        """
        from itertools import product

        # Generate all combinations
        param_names = list(param_grid.keys())
        param_values = list(param_grid.values())
        combinations = list(product(*param_values))

        print(f"Starting grid search with {len(combinations)} combinations...")
        print(f"Parameters: {param_names}\n")

        for i, param_combo in enumerate(combinations, 1):
            # Create config for this combination
            config = ExperimentConfig(**asdict(self.base_config))

            # Update parameters
            for param_name, param_value in zip(param_names, param_combo):
                setattr(config, param_name, param_value)

            # Set experiment name
            config.experiment_name = f"exp_{i:03d}_" + "_".join(
                f"{name}={value}" for name, value in zip(param_names, param_combo)
            )

            print(f"\n[{i}/{len(combinations)}] {config.experiment_name}")

            # Run experiment
            runner = ExperimentRunner(config)
            results = runner.run()
            self.results.append(results)

        # Convert to DataFrame
        results_df = self._results_to_dataframe()
        return results_df

    def _results_to_dataframe(self) -> pd.DataFrame:
        """Convert results to pandas DataFrame"""
        records = []

        for result in self.results:
            record = {
                'experiment_name': result['config']['experiment_name'],
                'timestamp': result['timestamp'],
                'duration_seconds': result['duration_seconds'],

                # Configuration
                'vocab_size': result['config']['vocab_size'],
                'knn_neighbors': result['config']['knn_neighbors'],
                'knn_metric': result['config']['knn_metric'],
                'image_max_size': result['config']['image_max_size'],
                'normalize_histograms': result['config']['normalize_histograms'],
                'use_minibatch_kmeans': result['config']['use_minibatch_kmeans'],

                # Metrics
                'accuracy': result['metrics']['accuracy'],
                'precision': result['metrics']['precision'],
                'recall': result['metrics']['recall'],
                'f1_score': result['metrics']['f1_score'],

                # Data stats
                'n_images': result['data_stats']['n_images'],
                'n_descriptors': result['data_stats']['n_descriptors'],
            }
            records.append(record)

        return pd.DataFrame(records)


In [9]:
class ResultsVisualizer:
    """Visualizes experiment results"""

    @staticmethod
    def plot_confusion_matrix(metrics: Dict, class_names: List[str],
                             experiment_name: str = "", figsize=(10, 8)):
        """Plot confusion matrix heatmap"""
        cm = metrics['confusion_matrix']

        plt.figure(figsize=figsize)
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                   xticklabels=class_names,
                   yticklabels=class_names)

        plt.title(f'Confusion Matrix\n{experiment_name}')
        plt.ylabel('True Label')
        plt.xlabel('Predicted Label')
        plt.tight_layout()
        plt.show()

    @staticmethod
    def plot_metrics_comparison(results_df: pd.DataFrame, figsize=(12, 6)):
        """Plot comparison of metrics across experiments"""
        metrics = ['accuracy', 'precision', 'recall', 'f1_score']

        fig, axes = plt.subplots(2, 2, figsize=figsize)
        axes = axes.ravel()

        for i, metric in enumerate(metrics):
            ax = axes[i]
            results_df.plot(x='experiment_name', y=metric,
                          kind='bar', ax=ax, legend=False)
            ax.set_title(metric.replace('_', ' ').title())
            ax.set_xlabel('')
            ax.set_ylabel('Score')
            ax.tick_params(axis='x', rotation=45)
            ax.grid(axis='y', alpha=0.3)

        plt.tight_layout()
        plt.show()

    @staticmethod
    def plot_parameter_impact(results_df: pd.DataFrame, param_name: str,
                             metric: str = 'accuracy', figsize=(10, 6)):
        """Plot impact of a specific parameter on performance"""
        plt.figure(figsize=figsize)

        grouped = results_df.groupby(param_name)[metric].agg(['mean', 'std'])

        plt.errorbar(grouped.index, grouped['mean'],
                    yerr=grouped['std'], marker='o', capsize=5)

        plt.title(f'Impact of {param_name} on {metric}')
        plt.xlabel(param_name)
        plt.ylabel(metric.replace('_', ' ').title())
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

    @staticmethod
    def plot_best_vs_worst(results_df: pd.DataFrame, metric: str = 'accuracy',
                          n_experiments: int = 5, figsize=(12, 6)):
        """Compare best and worst performing experiments"""
        sorted_df = results_df.sort_values(metric, ascending=False)

        best_df = sorted_df.head(n_experiments)
        worst_df = sorted_df.tail(n_experiments)

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=figsize)

        # Best experiments
        best_df.plot(x='experiment_name', y=metric, kind='barh',
                    ax=ax1, legend=False, color='green')
        ax1.set_title(f'Top {n_experiments} Experiments')
        ax1.set_xlabel(metric.title())

        # Worst experiments
        worst_df.plot(x='experiment_name', y=metric, kind='barh',
                     ax=ax2, legend=False, color='red')
        ax2.set_title(f'Bottom {n_experiments} Experiments')
        ax2.set_xlabel(metric.title())

        plt.tight_layout()
        plt.show()


In [10]:
class ExperimentManager:
    """Manages saving and loading experiment results"""

    def __init__(self, output_dir: str = "experiments"):
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(exist_ok=True)

    def save_results(self, results: Dict, filename: str = None):
        """Save experiment results to JSON"""
        if filename is None:
            filename = f"{results['config']['experiment_name']}.json"

        filepath = self.output_dir / filename

        # Convert numpy arrays to lists for JSON serialization
        results_serializable = self._make_serializable(results)

        with open(filepath, 'w') as f:
            json.dump(results_serializable, f, indent=2)

        print(f"Results saved to: {filepath}")

    def save_results_dataframe(self, df: pd.DataFrame, filename: str = "all_results.csv"):
        """Save results DataFrame to CSV"""
        filepath = self.output_dir / filename
        df.to_csv(filepath, index=False)
        print(f"Results DataFrame saved to: {filepath}")

    def load_results(self, filename: str) -> Dict:
        """Load experiment results from JSON"""
        filepath = self.output_dir / filename

        with open(filepath, 'r') as f:
            results = json.load(f)

        return results

    def _make_serializable(self, obj):
        """Convert numpy arrays and other non-serializable objects"""
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        elif isinstance(obj, dict):
            return {k: self._make_serializable(v) for k, v in obj.items()}
        elif isinstance(obj, list):
            return [self._make_serializable(item) for item in obj]
        elif isinstance(obj, (np.integer, np.floating)):
            return float(obj)
        else:
            return obj


In [ ]:
# Example 2: Hyperparameter tuning with grid search
# Define base configuration
base_config = ExperimentConfig(
data_dir="flowers",
experiment_description="Grid search for optimal parameters"
)

# Define parameter grid
param_grid = {
'vocab_size': [50, 100, 150, 200],
'knn_neighbors': [3, 5, 7, 9],
'normalize_histograms': [True, False]
}

# Run grid search
tuner = HyperparameterTuner(base_config)
results_df = tuner.tune(param_grid)

# Display results
print("\n" + "="*60)
print("GRID SEARCH RESULTS")
print("="*60)
print(results_df.sort_values('accuracy', ascending=False).head(10))

# Visualize
visualizer = ResultsVisualizer()
visualizer.plot_metrics_comparison(results_df)
visualizer.plot_parameter_impact(results_df, 'vocab_size')
visualizer.plot_parameter_impact(results_df, 'knn_neighbors')
visualizer.plot_best_vs_worst(results_df)

# Save results
manager = ExperimentManager()
manager.save_results_dataframe(results_df)

# Get best configuration
best_idx = results_df['accuracy'].idxmax()
best_config = results_df.loc[best_idx]
print("\n" + "="*60)
print("BEST CONFIGURATION")
print("="*60)
print(best_config)